# Rocket Launch Data ETL Process with API

This notebook implements an ETL process for upcoming rocket launch data using a public API.

## ETL Summary
**Extract**
- Request upcoming launch data from the SpaceDevs API

**Transform**
- Convert the JSON response into a pandas DataFrame
- Extract image URLs from launch records

**Load**
- Save the API response to `data/launches.json`
- Download launch images into the `image` folder

## Additional Task
- Load and save the Titanic dataset as a CSV file

In [ ]:
import os
import json
import requests
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

## Step 1. Load data from API and save JSON

In [ ]:
def load_data_from_api():
    url = "https://ll.thespacedevs.com/2.0.0/launch/upcoming"

    # Create directory for data storage
    os.makedirs("data", exist_ok=True)

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        # Save API response to a JSON file
        with open("data/launches.json", "w", encoding="utf-8") as file:
            json.dump(data, file, ensure_ascii=False, indent=4)

        print("JSON file saved: data/launches.json")

        # Extract launch list and convert to DataFrame
        launches = data.get("results", [])
        df = pd.json_normalize(launches)

        return df

    except requests.exceptions.RequestException as error:
        print("API request error:", error)
        return pd.DataFrame()

In [ ]:
df = load_data_from_api()
df.head()

## Check available columns

In [ ]:
print(df.columns.tolist())

## Preview selected columns

In [ ]:
selected_columns = ["name", "net", "status.name", "image"]
available_columns = [column for column in selected_columns if column in df.columns]

if available_columns:
    display(df[available_columns].head())
else:
    print("Selected columns are not available in the DataFrame.")

## Step 2. Read saved JSON, extract image URLs, and download images

In [ ]:
def get_pictures(limit=10):
    # Create directory for images
    os.makedirs("image", exist_ok=True)

    try:
        with open("data/launches.json", "r", encoding="utf-8") as file:
            data = json.load(file)

        launches = data.get("results", [])
        count = 0

        for index, launch in enumerate(launches):
            if count >= limit:
                break

            image_url = launch.get("image")
            launch_name = launch.get("name", f"launch_{index + 1}")

            if not image_url:
                print(f"[{index + 1}] Image not available: {launch_name}")
                continue

            try:
                response = requests.get(image_url, timeout=10)
                response.raise_for_status()

                extension = image_url.split(".")[-1].split("?")[0].lower()
                if extension not in ["jpg", "jpeg", "png", "webp"]:
                    extension = "jpg"

                file_path = f"image/launch_{count + 1}.{extension}"

                with open(file_path, "wb") as image_file:
                    image_file.write(response.content)

                print(f"Saved: {file_path}")
                count += 1

            except requests.exceptions.RequestException as error:
                print(f"Image download error for {launch_name}: {error}")

        print(f"Total images downloaded: {count}")

    except FileNotFoundError:
        print("The file data/launches.json was not found. Run load_data_from_api() first.")
    except json.JSONDecodeError:
        print("JSON decoding error.")

In [ ]:
get_pictures(limit=10)

## Step 3. Read and visualize downloaded images

In [ ]:
def show_images(max_images=5):
    folder = "image"

    if not os.path.exists(folder):
        print("The folder 'image' does not exist.")
        return

    files = os.listdir(folder)
    image_files = [
        file_name for file_name in files
        if file_name.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
    ]

    if not image_files:
        print("No images found.")
        return

    for file_name in image_files[:max_images]:
        file_path = os.path.join(folder, file_name)

        try:
            image = Image.open(file_path)
            plt.figure(figsize=(6, 6))
            plt.imshow(image)
            plt.title(file_name)
            plt.axis("off")
            plt.show()

        except Exception as error:
            print(f"Error opening {file_name}: {error}")

In [ ]:
show_images(max_images=5)

## Step 4. Additional ETL task: Titanic dataset

In [ ]:
def etl_titanic():
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

    # Create directory for data storage
    os.makedirs("data", exist_ok=True)

    try:
        df = pd.read_csv(url)
        df.to_csv("data/titanic.csv", index=False, encoding="utf-8-sig")

        print("Titanic CSV saved: data/titanic.csv")
        print("Data shape:", df.shape)

        return df

    except Exception as error:
        print("Error loading Titanic dataset:", error)
        return pd.DataFrame()

In [ ]:
titanic_df = etl_titanic()
titanic_df.head()